# Kiểm tra các đề xuất nâng cấp CRISP-DM

Notebook thực hiện các kiểm tra có thể tái lập trên pipeline POC hiện tại: ECE/MCE, biến động thứ hạng, thử nghiệm trend không chồng lấn và power analysis minh họa. Dùng train/validation; **không đọc test**. Các kết quả thống kê/kinh doanh có điều kiện và không khẳng định tác động của can thiệp.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm,spearmanr
from scipy.optimize import brentq
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
DATA=ROOT/'artifacts'/'data_preparation';OUT=ROOT/'artifacts'/'research'/'crisp_dm_review'
train,features=load_prepared_split(DATA,'train');val,vfeatures=load_prepared_split(DATA,'validation')
assert features==vfeatures and train.snapshot_day.max()+7 < val.snapshot_day.min()
pred=pd.read_parquet(ROOT/'artifacts'/'calibration_final_poc'/'predictions.parquet',filters=[('split','=','validation')])
assert set(pred.feature_set)=={'18 feature','18 + chu kỳ'}
assert len(pred)==2*len(val) and not pred.duplicated(['feature_set','household_key','snapshot_day']).any()
print('Train:',len(train),'| Validation:',len(val),'| Test không được đọc.')

Train: 129834 | Validation: 27480 | Test không được đọc.


## 1. Calibration: ECE, MCE và độ nhạy theo số bin

ECE là sai lệch tuyệt đối trung bình có trọng số giữa xác suất và tỷ lệ quan sát trong bin; MCE là sai lệch lớn nhất. Các chỉ số phụ thuộc cách chia bin và cỡ mẫu; MCE của bin nhỏ dễ biến động. Brier, log loss và reliability diagram vẫn cần được đọc cùng ECE. Household lặp lại qua các cutoff.

In [2]:
def calibration_errors(y,p,n_bins):
    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
    if len(y)==0 or len(y)!=len(p) or not np.isfinite(p).all() or not np.isin(y,[0,1]).all() or ((p<0)|(p>1)).any():
        raise ValueError('Dữ liệu nhãn/xác suất không hợp lệ')
    edges=np.linspace(0,1,n_bins+1);index=np.minimum(np.searchsorted(edges,p,side='right')-1,n_bins-1)
    rows=[]
    for i in range(n_bins):
        mask=index==i
        if not mask.any():continue
        gap=abs(y[mask].mean()-p[mask].mean())
        rows.append({'bin':i,'lower':edges[i],'upper':edges[i+1],'n':int(mask.sum()),
                     'observed':y[mask].mean(),'predicted':p[mask].mean(),'gap':gap})
    table=pd.DataFrame(rows)
    return {'ece':float((table.n*table.gap).sum()/len(y)),'mce':float(table.gap.max()),'n_bins_nonempty':len(table)},table
assert np.isclose(calibration_errors([0,1],[0,1],2)[0]['ece'],0)
ece_rows=[]
for name,g in pred.groupby('feature_set'):
    for n_bins in [10,15,20]:ece_rows.append({'feature_set':name,'bins':n_bins,**calibration_errors(g.target_any_basket,g.probability,n_bins)[0]})
ece_table=pd.DataFrame(ece_rows);display(ece_table)

,feature_set,bins,ece,mce,n_bins_nonempty
0,18 + chu kỳ,10,0.016250,0.045817,10
1,18 + chu kỳ,15,0.017203,0.046038,15
2,18 + chu kỳ,20,0.019954,0.049110,20
3,18 feature,10,0.017030,0.041650,10
4,18 feature,15,0.017140,0.041719,15
5,18 feature,20,0.017348,0.042422,20


## 2. Biến động thứ hạng qua các cutoff

Spearman tính giữa xác suất của các household xuất hiện ở cả hai cutoff liên tiếp; top-10% overlap dùng thứ hạng giảm dần và phá đồng điểm theo household ID. Thứ hạng thay đổi có thể phản ánh hành vi thay đổi, không tự chứng minh model không ổn định. Không áp dụng ngưỡng 0,85/0,70 khi chưa có căn cứ nghiệp vụ.

In [3]:
rank_rows=[]
for name,g in pred.groupby('feature_set'):
    cutoffs=sorted(g.snapshot_day.unique())
    for a,b in zip(cutoffs,cutoffs[1:]):
        left=g[g.snapshot_day.eq(a)][['household_key','probability']]
        right=g[g.snapshot_day.eq(b)][['household_key','probability']]
        both=left.merge(right,on='household_key',validate='one_to_one',suffixes=('_a','_b'))
        if len(both)<10:continue
        correlation=spearmanr(both.probability_a,both.probability_b).statistic
        k=int(np.ceil(.1*len(both)))
        one=set(both.sort_values(['probability_a','household_key'],ascending=[False,True]).head(k).household_key)
        two=set(both.sort_values(['probability_b','household_key'],ascending=[False,True]).head(k).household_key)
        rank_rows.append({'feature_set':name,'cutoff_a':a,'cutoff_b':b,'common_households':len(both),
                          'spearman':correlation,'top10_overlap':len(one&two)/k})
ranks=pd.DataFrame(rank_rows);display(ranks)
display(ranks.groupby('feature_set')[['spearman','top10_overlap']].agg(['mean','min','max']))

,feature_set,cutoff_a,cutoff_b,common_households,spearman,top10_overlap
0,18 + chu kỳ,551,558,2498,0.977476,0.852
1,18 + chu kỳ,558,565,2498,0.975436,0.864
2,18 + chu kỳ,565,572,2498,0.977202,0.864
3,18 + chu kỳ,572,579,2498,0.976453,0.860
4,18 + chu kỳ,579,586,2498,0.977115,0.856
5,18 + chu kỳ,586,593,2498,0.975838,0.848
6,18 + chu kỳ,593,600,2498,0.976835,0.824
7,18 + chu kỳ,600,607,2498,0.976771,0.844
8,18 + chu kỳ,607,614,2498,0.975541,0.828
9,18 + chu kỳ,614,621,2499,0.979317,0.888


spearman                     top10_overlap              
                 mean       min       max          mean    min    max
feature_set                                                          
18 + chu kỳ  0.976798  0.975436  0.979317        0.8528  0.824  0.888
18 feature   0.977240  0.975784  0.979215        0.8364  0.816  0.868

## 3. Trend theo các khoảng không chồng lấn

So sánh 7 ngày gần nhất với **23 ngày trước đó**, chuẩn hóa số basket hoặc sales theo ngày. Trend dài hạn so sánh 90 ngày gần nhất với **90 ngày trước đó**. Với hộ có ít hơn 30 hoặc 180 ngày tính từ lần đầu ghi nhận, trend tương ứng để thiếu và thêm cờ đủ lịch sử; đủ thời gian quan sát không bảo đảm dữ liệu nguồn hoàn chỉnh. `log1p` trên trung bình mỗi ngày tránh chia cho giá trị gần 0. Chỉ là thí nghiệm feature, không khẳng định feature gây ra hành vi.

In [4]:
def add_trend(frame):
    out=frame.copy()
    out['trend_has_30d_history']=(out.observed_tenure_days+1>=30).astype(int)
    out['trend_has_180d_history']=(out.observed_tenure_days+1>=180).astype(int)
    cols=['trend_has_30d_history','trend_has_180d_history']
    for metric in ['baskets','sales']:
        recent=out[f'{metric}_7d']/7
        prior=(out[f'{metric}_30d']-out[f'{metric}_7d'])/23
        if (prior < -1e-8).any():raise ValueError('Tổng prior 23 ngày âm; cần kiểm tra nguồn')
        col=f'{metric}_trend_7_vs_previous23'
        out[col]=np.where(out.trend_has_30d_history.eq(1),np.log1p(recent)-np.log1p(prior.clip(lower=0)),np.nan)
        cols.append(col)
    for metric in ['baskets','sales','active_days']:
        recent=out[f'{metric}_90d']/90
        prior=(out[f'{metric}_180d']-out[f'{metric}_90d'])/90
        if (prior < -1e-8).any():raise ValueError('Tổng prior 90 ngày âm; cần kiểm tra nguồn')
        col=f'{metric}_trend_90_vs_previous90'
        out[col]=np.where(out.trend_has_180d_history.eq(1),np.log1p(recent)-np.log1p(prior.clip(lower=0)),np.nan)
        cols.append(col)
    return out,cols
tr,trend_cols=add_trend(train);va,trend_cols_val=add_trend(val)
assert trend_cols==trend_cols_val and not va[trend_cols].isin([np.inf,-np.inf]).any().any()
display(tr[trend_cols].isna().mean().rename('tỷ lệ thiếu train').to_frame())

,tỷ lệ thiếu train
trend_has_30d_history,0.000000
trend_has_180d_history,0.000000
baskets_trend_7_vs_previous23,0.000131
sales_trend_7_vs_previous23,0.000131
baskets_trend_90_vs_previous90,0.185082
sales_trend_90_vs_previous90,0.185082
active_days_trend_90_vs_previous90,0.185082


## 4. Thử trend so với 18 feature

Giữ cấu hình Logistic Regression và LightGBM trước đây; mỗi phiên bản được fit độc lập trên train. Logistic học imputer/scaler chỉ trên train, LightGBM nhận missing. So sánh này tiếp tục dùng validation nên có nguy cơ chọn quá khớp validation. Chưa kết hợp với cadence, chưa tuning và không dùng test.

In [5]:
def weekly_lift(frame,p):
    values=[]
    for ix in frame.groupby('snapshot_day').indices.values():
        y=frame.target_any_basket.to_numpy()[ix];q=p[ix];k=int(np.ceil(.1*len(y)))
        threshold=np.partition(q,len(q)-k)[len(q)-k];high=q>threshold;ties=q==threshold
        precision=(y[high].sum()+(k-high.sum())*y[ties].mean())/k
        values.append(precision/y.mean() if y.mean() else np.nan)
    return np.array(values)
rows=[];weekly=[]
for experiment,extra in [('18 feature',[]),('18 + trend',trend_cols)]:
    for name in ['Logistic Regression','LightGBM']:
        cols=features+extra
        model=(make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),StandardScaler(),LogisticRegression(max_iter=2000,random_state=42)) if name=='Logistic Regression'
               else LGBMClassifier(n_estimators=200,learning_rate=.05,num_leaves=15,min_child_samples=100,random_state=42,verbosity=-1,n_jobs=2))
        model.fit(tr[cols],tr.target_any_basket);p=model.predict_proba(va[cols])[:,1];lifts=weekly_lift(va,p)
        rows.append({'model':name,'experiment':experiment,'brier':brier_score_loss(va.target_any_basket,p),
                     'log_loss':log_loss(va.target_any_basket,p,labels=[0,1]),'roc_auc':roc_auc_score(va.target_any_basket,p),
                     'weekly_lift_mean':lifts.mean()})
        for cutoff,value in zip(sorted(va.snapshot_day.unique()),lifts):weekly.append({'model':name,'experiment':experiment,'cutoff':cutoff,'lift':value})
trend_results=pd.DataFrame(rows);trend_weekly=pd.DataFrame(weekly)
display(trend_results)
ref=trend_weekly[trend_weekly.experiment.eq('18 feature')][['model','cutoff','lift']].rename(columns={'lift':'base_lift'})
stability=trend_weekly[trend_weekly.experiment.eq('18 + trend')].merge(ref,on=['model','cutoff'],validate='one_to_one')
stability['delta_lift']=stability.lift-stability.base_lift
display(stability.groupby('model').delta_lift.agg(['mean','min','max',lambda x:int((x>1e-10).sum())]))

,model,experiment,brier,log_loss,roc_auc,weekly_lift_mean
0,Logistic Regression,18 feature,0.166371,0.501671,0.833017,1.777743
1,LightGBM,18 feature,0.164389,0.495405,0.836662,1.786582
2,Logistic Regression,18 + trend,0.166281,0.501486,0.833197,1.777724
3,LightGBM,18 + trend,0.164359,0.495249,0.836698,1.783194


,mean,min,max,<lambda_0>
model,,,,
LightGBM,-0.003388,-0.014743,0.022041,2
Logistic Regression,-0.000019,-0.007451,0.007372,2


## 5. Power analysis minh họa — cần A/B thực tế

Ví dụ dùng baseline 0,5322 theo đề xuất và giả định uplift **tương đối 10%** (từ 53,22% lên 58,542%, tương ứng 5,322 điểm phần trăm), kiểm định hai phía alpha 0,05 và power 80%. Công thức xấp xỉ normal cho hai tỷ lệ, chia đều, household độc lập. Đây là phép thiết kế giả định, không thể suy ra tác động chiến dịch từ dữ liệu quan sát.

Với 1.250 hộ mỗi nhóm, MDE dưới đây tính theo cùng xấp xỉ và bao gồm cả thành phần power 80%; không dùng chỉ biên sai số 95%. Nếu chọn top propensity thay vì toàn dân số, baseline cần được ước lượng cho nhóm đủ điều kiện và thiết kế phải tính lặp liên hệ, cụm, rơi mẫu và các giả định khác. 2.500 hộ trong dataset không chứng minh có đủ đối tượng thử nghiệm thực tế.

In [6]:
p0=.5322; p1=p0*1.10; alpha=.05; power=.80
za=norm.ppf(1-alpha/2);zb=norm.ppf(power)
def required_per_arm(p0,p1):
    avg=(p0+p1)/2
    return (za*np.sqrt(2*avg*(1-avg))+zb*np.sqrt(p0*(1-p0)+p1*(1-p1)))**2/(p1-p0)**2
sample_size=int(np.ceil(required_per_arm(p0,p1)))
delta=brentq(lambda d:required_per_arm(p0,p0+d)-1250,1e-5,.3)
power_example=pd.DataFrame([{'control_rate_assumption':p0,'treatment_rate_assumption':p1,
                             'absolute_effect_pp':100*(p1-p0),'n_per_arm_approx':sample_size,
                             'total_approx':sample_size*2,'mde_pp_at_1250_per_arm':100*delta}])
display(power_example)

,control_rate_assumption,treatment_rate_assumption,absolute_effect_pp,n_per_arm_approx,total_approx,mde_pp_at_1250_per_arm
0,0.5322,0.58542,5.322,1366,2732,5.560058


### 5.1. Kiểm tra giả định baseline 15% so với 53,22%

Cùng mức tăng tương đối 10%, baseline 15% tương ứng 15% → 16,5%, tức tăng 1,5 điểm phần trăm; baseline 53,22% tương ứng tăng 5,322 điểm phần trăm. Cỡ mẫu khác nhau đáng kể. Đây là xấp xỉ hai tỷ lệ, alpha hai phía 0,05, power 80%, chia đều và hộ độc lập; chưa tính mất mẫu hay thiết kế lặp/cụm. Không kết luận cỡ mẫu pilot thực tế nếu chưa biết đối tượng và tỷ lệ đối chứng.

In [ ]:
power_scenarios=[]
for baseline in [.5322,.15]:
    treatment=baseline*1.10
    per_arm=int(np.ceil(required_per_arm(baseline,treatment)))
    mde=brentq(lambda d:required_per_arm(baseline,baseline+d)-1250,1e-5,.3)
    power_scenarios.append({'baseline_giả_định':baseline,'treatment_giả_định':treatment,
                            'uplift_tương_đối':.10,'uplift_điểm_%':100*(treatment-baseline),
                            'hộ_mỗi_nhóm_xấp_xỉ':per_arm,'tổng_hộ_xấp_xỉ':2*per_arm,
                            'MDE_điểm_%_tại_1250_mỗi_nhóm':100*mde})
power_scenarios=pd.DataFrame(power_scenarios)
display(power_scenarios)
OUT.mkdir(parents=True,exist_ok=True)
power_scenarios.to_csv(OUT/'power_scenarios_baseline.csv',index=False,encoding='utf-8-sig')

,baseline_giả_định,treatment_giả_định,uplift_tương_đối,uplift_điểm_%,hộ_mỗi_nhóm_xấp_xỉ,tổng_hộ_xấp_xỉ,MDE_điểm_%_tại_1250_mỗi_nhóm
0,0.5322,0.58542,0.1,5.322,1366,2732,5.560058
1,0.1500,0.16500,0.1,1.500,9257,18514,4.218188


## 6. Các đề xuất cần diễn giải đúng

- Khác biệt số buyer được xếp hạng so với chọn ngẫu nhiên **không phải uplift do tiếp xúc**; không suy ROI từ đó.
- `causal_data` mô tả product–store–week; sản phẩm được mua khi có display/mailer không xác nhận household thực sự tiếp xúc. Mã `0` là không có; dữ liệu thiếu không được đếm là exposure; cần kiểm tra duy nhất khóa và tính sẵn có tại cutoff.
- Không chọn feature chỉ vì một lần permutation importance âm, nhất là từ test; cần ablation/rolling validation. `has_demographics` là cờ hồ sơ, không phải thuộc tính được bảo vệ.
- Isotonic và Platt phải fit trên tập calibration và so sánh trên tập khác theo thời gian. ECE không tự động tốt hơn Brier/log loss.
- `DAY % 7` chỉ là vị trí ngày tương đối, không thể gọi là thứ trong tuần khi không biết mốc lịch. Xem [báo cáo rà soát](../../docs/research/RA_SOAT_DE_XUAT_NANG_CAP_CRISP_DM.md) cho từng đề xuất.

In [7]:
OUT.mkdir(parents=True,exist_ok=True)
ece_table.to_csv(OUT/'validation_ece_bins.csv',index=False,encoding='utf-8-sig')
ranks.to_csv(OUT/'validation_rank_stability.csv',index=False,encoding='utf-8-sig')
trend_results.to_csv(OUT/'trend_ablation_validation.csv',index=False,encoding='utf-8-sig')
stability.to_csv(OUT/'trend_weekly_lift.csv',index=False,encoding='utf-8-sig')
power_example.to_csv(OUT/'power_illustration.csv',index=False,encoding='utf-8-sig')
print('Đã lưu:',OUT)

Đã lưu: artifacts\crisp_dm_review
